# Capstone 4 — Agentic ETL Data Pipeline

**Book:** [Agentic AI: Build, Ship, Portfolio](../index.html)

---

In this capstone you will build an **agentic ETL pipeline** that:

1. Infers schema from raw data automatically.
2. Validates data quality with configurable rules.
3. Applies intelligent transformations via an LLM-guided agent.
4. Self-heals on failure (retries with corrective actions).
5. Orchestrates the full pipeline with monitoring.
6. Provides a monitoring dashboard with metrics.

> **Note:** This notebook uses *mock/simulated* LLM responses so it runs without an API key. Replace the mock functions with real LLM calls for production use.

---
## 1 Setup

In [ ]:
# --- Setup ---

import csv
import io
import json
import re
import time
import traceback
import uuid
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any

print("All imports ready.")

In [ ]:
# --- Mock LLM Helper ---

def mock_llm(system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
    """Simulate an LLM response based on keywords in the prompt."""
    lower = (system_prompt + user_prompt).lower()
    if "schema" in lower and ("infer" in lower or "detect" in lower):
        return json.dumps({
            "columns": [
                {"name": "id", "type": "integer", "nullable": False, "description": "Unique identifier"},
                {"name": "name", "type": "string", "nullable": False, "description": "Full name"},
                {"name": "email", "type": "string", "nullable": False, "description": "Email address"},
                {"name": "amount", "type": "float", "nullable": True, "description": "Transaction amount in USD"},
                {"name": "date", "type": "date", "nullable": False, "description": "Transaction date"}
            ],
            "row_count_estimate": 100,
            "delimiter": ",",
            "has_header": True
        })
    elif "transform" in lower or "clean" in lower:
        return json.dumps({
            "transformations": [
                {"column": "name", "action": "strip_whitespace", "description": "Remove leading/trailing spaces"},
                {"column": "email", "action": "lowercase", "description": "Normalize to lowercase"},
                {"column": "amount", "action": "fill_null", "value": "0.0", "description": "Replace nulls with 0.0"},
                {"column": "date", "action": "parse_date", "format": "%Y-%m-%d", "description": "Parse to standard format"}
            ]
        })
    elif "fix" in lower or "heal" in lower or "repair" in lower:
        return json.dumps({
            "diagnosis": "The pipeline failed due to malformed date values in 3 rows.",
            "fix_strategy": "Replace malformed dates with the most common date format found in valid rows.",
            "confidence": 0.85,
            "affected_rows": [5, 12, 23]
        })
    elif "quality" in lower or "validat" in lower:
        return json.dumps({
            "overall_quality": 0.82,
            "issues": [
                {"rule": "not_null", "column": "email", "violations": 2, "severity": "error"},
                {"rule": "format", "column": "email", "violations": 3, "severity": "warning"},
                {"rule": "range", "column": "amount", "violations": 1, "severity": "warning"}
            ]
        })
    else:
        return "Mock response: Pipeline step completed."

print("Mock LLM helper ready.")

In [ ]:
# --- Sample Data ---

SAMPLE_CSV = """id,name,email,amount,date
1,Alice Johnson,alice@example.com,150.00,2026-01-15
2, Bob Smith ,BOB@EXAMPLE.COM,200.50,2026-01-16
3,Charlie Brown,charlie@example.com,,2026-01-17
4,Diana Prince,diana@example,175.25,2026-01-18
5,Eve Wilson,eve@example.com,-50.00,not-a-date
6,Frank Castle,,300.00,2026-01-20
7,Grace Hopper,grace@example.com,450.75,2026-01-21
8,Hank Pym,hank@example.com,99.99,2026/01/22
9,Ivy Chen,ivy@example.com,1250.00,2026-01-23
10,Jack Ryan,jack@example.com,75.50,2026-01-24
""".strip()

def parse_csv(csv_text: str) -> list[dict[str, str]]:
    """Parse CSV text into a list of dicts."""
    reader = csv.DictReader(io.StringIO(csv_text))
    return [dict(row) for row in reader]

sample_data = parse_csv(SAMPLE_CSV)
print(f"Sample data: {len(sample_data)} rows, columns: {list(sample_data[0].keys())}")
for row in sample_data[:3]:
    print(f"  {row}")

---
## 2 Schema Inference Agent

The **Schema Inference Agent** examines raw data and infers a structured schema including column types, nullability, and descriptions.

In [ ]:
@dataclass
class ColumnSchema:
    """Schema for a single column."""
    name: str
    inferred_type: str
    nullable: bool
    description: str
    sample_values: list[str] = field(default_factory=list)
    null_count: int = 0
    unique_count: int = 0


@dataclass
class DataSchema:
    """Complete inferred schema."""
    columns: list[ColumnSchema]
    row_count: int
    inferred_at: str = field(default_factory=lambda: datetime.now().isoformat())


class SchemaInferenceAgent:
    """Infers data schema from raw data."""

    TYPE_PATTERNS = {
        "integer": re.compile(r'^-?\d+$'),
        "float": re.compile(r'^-?\d+\.\d+$'),
        "date": re.compile(r'^\d{4}[-/]\d{2}[-/]\d{2}$'),
        "email": re.compile(r'^[^@]+@[^@]+\.[^@]+$'),
        "boolean": re.compile(r'^(true|false|yes|no|0|1)$', re.IGNORECASE),
    }

    def _infer_type(self, values: list[str]) -> str:
        """Infer column type from sample values."""
        non_empty = [v for v in values if v.strip()]
        if not non_empty:
            return "string"

        type_counts: dict[str, int] = Counter()
        for v in non_empty:
            matched = False
            for type_name, pattern in self.TYPE_PATTERNS.items():
                if pattern.match(v.strip()):
                    type_counts[type_name] += 1
                    matched = True
                    break
            if not matched:
                type_counts["string"] += 1

        # Return the most common type
        if type_counts:
            return type_counts.most_common(1)[0][0]
        return "string"

    def infer(self, data: list[dict[str, str]]) -> DataSchema:
        """Infer schema from a list of row dicts."""
        if not data:
            return DataSchema(columns=[], row_count=0)

        columns = []
        col_names = list(data[0].keys())

        for col in col_names:
            values = [row.get(col, "") for row in data]
            non_empty = [v for v in values if v.strip()]
            null_count = len(values) - len(non_empty)
            unique_count = len(set(non_empty))

            col_schema = ColumnSchema(
                name=col,
                inferred_type=self._infer_type(values),
                nullable=null_count > 0,
                description=f"Column '{col}'",
                sample_values=non_empty[:5],
                null_count=null_count,
                unique_count=unique_count
            )
            columns.append(col_schema)

        # Enhance descriptions with LLM
        llm_raw = mock_llm(
            system_prompt="You are a data schema inference agent.",
            user_prompt=f"Infer the schema for this data: columns={col_names}, "
                        f"sample={data[:3]}"
        )
        llm_schema = json.loads(llm_raw)
        for llm_col in llm_schema.get("columns", []):
            for col in columns:
                if col.name == llm_col["name"]:
                    col.description = llm_col.get("description", col.description)

        schema = DataSchema(columns=columns, row_count=len(data))
        print(f"[SchemaInference] Inferred schema: {len(columns)} columns, {len(data)} rows")
        return schema

print("SchemaInferenceAgent defined.")

In [ ]:
# --- Test Schema Inference ---

schema_agent = SchemaInferenceAgent()
schema = schema_agent.infer(sample_data)

print(f"\nInferred Schema ({schema.row_count} rows):")
for col in schema.columns:
    print(f"  {col.name:12s} | type={col.inferred_type:8s} | "
          f"nullable={col.nullable} | nulls={col.null_count} | "
          f"unique={col.unique_count} | {col.description}")

---
## 3 Data Quality Checker

The **Data Quality Checker** validates data against configurable rules: null checks, format validation, range checks, uniqueness constraints, and referential integrity.

In [ ]:
class QualitySeverity(Enum):
    ERROR = "error"
    WARNING = "warning"
    INFO = "info"


@dataclass
class QualityRule:
    """A single data quality rule."""
    name: str
    column: str
    rule_type: str  # not_null, format, range, unique
    params: dict[str, Any] = field(default_factory=dict)
    severity: QualitySeverity = QualitySeverity.ERROR


@dataclass
class QualityViolation:
    """A single quality violation."""
    rule: QualityRule
    row_index: int
    value: str
    message: str


@dataclass
class QualityReport:
    """Complete quality report."""
    total_rows: int
    violations: list[QualityViolation]
    quality_score: float  # 0.0 to 1.0
    passed: bool
    summary: dict[str, int]  # rule_type -> count


class DataQualityChecker:
    """Validates data against quality rules."""

    def __init__(self, rules: list[QualityRule], fail_threshold: float = 0.7):
        self.rules = rules
        self.fail_threshold = fail_threshold

    def _check_not_null(self, value: str, rule: QualityRule) -> str | None:
        if not value.strip():
            return f"Column '{rule.column}' is null or empty."
        return None

    def _check_format(self, value: str, rule: QualityRule) -> str | None:
        if not value.strip():
            return None  # Skip nulls (handled by not_null)
        pattern = rule.params.get("pattern", ".*")
        if not re.match(pattern, value.strip()):
            return f"Column '{rule.column}' value '{value}' does not match pattern '{pattern}'."
        return None

    def _check_range(self, value: str, rule: QualityRule) -> str | None:
        if not value.strip():
            return None
        try:
            num = float(value)
            min_val = rule.params.get("min", float("-inf"))
            max_val = rule.params.get("max", float("inf"))
            if num < min_val or num > max_val:
                return f"Column '{rule.column}' value {num} outside range [{min_val}, {max_val}]."
        except ValueError:
            return f"Column '{rule.column}' value '{value}' is not numeric."
        return None

    def check(self, data: list[dict[str, str]]) -> QualityReport:
        """Run all quality rules against the data."""
        violations: list[QualityViolation] = []

        for rule in self.rules:
            for i, row in enumerate(data):
                value = row.get(rule.column, "")
                msg = None

                if rule.rule_type == "not_null":
                    msg = self._check_not_null(value, rule)
                elif rule.rule_type == "format":
                    msg = self._check_format(value, rule)
                elif rule.rule_type == "range":
                    msg = self._check_range(value, rule)

                if msg:
                    violations.append(QualityViolation(
                        rule=rule, row_index=i, value=value, message=msg
                    ))

        total_checks = len(self.rules) * len(data)
        quality_score = 1.0 - (len(violations) / max(total_checks, 1))
        quality_score = round(max(quality_score, 0.0), 3)

        summary: dict[str, int] = Counter()
        for v in violations:
            summary[v.rule.rule_type] += 1

        report = QualityReport(
            total_rows=len(data),
            violations=violations,
            quality_score=quality_score,
            passed=quality_score >= self.fail_threshold,
            summary=dict(summary)
        )

        print(f"[QualityChecker] Score: {quality_score:.3f}, "
              f"Violations: {len(violations)}, Passed: {report.passed}")
        return report

print("DataQualityChecker defined.")

In [ ]:
# --- Define quality rules and test ---

QUALITY_RULES = [
    QualityRule(name="email_not_null", column="email", rule_type="not_null",
                severity=QualitySeverity.ERROR),
    QualityRule(name="email_format", column="email", rule_type="format",
                params={"pattern": r"^[^@]+@[^@]+\.[^@]+$"},
                severity=QualitySeverity.WARNING),
    QualityRule(name="amount_range", column="amount", rule_type="range",
                params={"min": 0, "max": 10000},
                severity=QualitySeverity.WARNING),
    QualityRule(name="date_format", column="date", rule_type="format",
                params={"pattern": r"^\d{4}-\d{2}-\d{2}$"},
                severity=QualitySeverity.ERROR),
    QualityRule(name="name_not_null", column="name", rule_type="not_null",
                severity=QualitySeverity.ERROR),
]

quality_checker = DataQualityChecker(rules=QUALITY_RULES, fail_threshold=0.7)
quality_report = quality_checker.check(sample_data)

print(f"\nQuality Score: {quality_report.quality_score:.3f}")
print(f"Passed: {quality_report.passed}")
print(f"Summary: {quality_report.summary}")
print(f"\nViolations:")
for v in quality_report.violations:
    print(f"  Row {v.row_index}: [{v.rule.severity.value}] {v.message}")

---
## 4 Transform Agent

The **Transform Agent** applies data transformations based on the schema and quality report. It uses the LLM to suggest transformations and then applies them.

In [ ]:
@dataclass
class TransformAction:
    """A single transformation action."""
    column: str
    action: str
    description: str
    params: dict[str, Any] = field(default_factory=dict)
    rows_affected: int = 0


@dataclass
class TransformResult:
    """Result of transformation."""
    data: list[dict[str, str]]
    actions_applied: list[TransformAction]
    rows_modified: int
    success: bool
    error: str | None = None


class TransformAgent:
    """Applies intelligent data transformations."""

    TRANSFORM_FUNCTIONS = {
        "strip_whitespace": lambda v: v.strip(),
        "lowercase": lambda v: v.lower(),
        "uppercase": lambda v: v.upper(),
        "fill_null": None,  # Handled specially
        "parse_date": None,  # Handled specially
    }

    def _suggest_transforms(self, schema: DataSchema,
                            quality_report: QualityReport) -> list[dict]:
        """Use LLM to suggest transformations."""
        raw = mock_llm(
            system_prompt="You are a data transformation agent.",
            user_prompt=(
                f"Suggest transformations for data with schema: "
                f"{[c.name for c in schema.columns]} and "
                f"{len(quality_report.violations)} quality issues."
            )
        )
        data = json.loads(raw)
        return data.get("transformations", [])

    def _apply_transform(self, value: str, action: str,
                         params: dict[str, Any]) -> str:
        """Apply a single transform to a value."""
        if action == "fill_null" and not value.strip():
            return params.get("value", "")
        elif action == "parse_date":
            # Normalize date separators
            return value.replace("/", "-")
        elif action in self.TRANSFORM_FUNCTIONS:
            fn = self.TRANSFORM_FUNCTIONS[action]
            if fn and value.strip():
                return fn(value)
        return value

    def transform(self, data: list[dict[str, str]], schema: DataSchema,
                  quality_report: QualityReport) -> TransformResult:
        """Apply transformations to the data."""
        suggested = self._suggest_transforms(schema, quality_report)

        actions: list[TransformAction] = []
        transformed_data = [dict(row) for row in data]  # Deep copy
        total_modified = 0

        for t in suggested:
            col = t["column"]
            action = t["action"]
            desc = t.get("description", "")
            params = {k: v for k, v in t.items() if k not in ("column", "action", "description")}
            rows_affected = 0

            for row in transformed_data:
                if col in row:
                    old_val = row[col]
                    new_val = self._apply_transform(old_val, action, params)
                    if old_val != new_val:
                        row[col] = new_val
                        rows_affected += 1

            total_modified += rows_affected
            actions.append(TransformAction(
                column=col, action=action, description=desc,
                params=params, rows_affected=rows_affected
            ))

        result = TransformResult(
            data=transformed_data,
            actions_applied=actions,
            rows_modified=total_modified,
            success=True
        )

        print(f"[TransformAgent] {len(actions)} transforms applied, "
              f"{total_modified} modifications")
        return result

print("TransformAgent defined.")

In [ ]:
# --- Test Transform Agent ---

transform_agent = TransformAgent()
transform_result = transform_agent.transform(sample_data, schema, quality_report)

print(f"\nTransform Actions:")
for a in transform_result.actions_applied:
    print(f"  {a.column}.{a.action}: {a.description} ({a.rows_affected} rows)")

print(f"\nBefore vs After (first 3 rows):")
for i in range(3):
    print(f"  Before: {sample_data[i]}")
    print(f"  After:  {transform_result.data[i]}")
    print()

---
## 5 Self-Healing Pipeline (Retry/Fix Logic)

The **Self-Healing Pipeline** wraps each step with error detection, diagnosis, and automatic fix attempts. If a step fails, the agent analyzes the error, suggests a fix, and retries.

In [ ]:
@dataclass
class HealingAttempt:
    """Record of a self-healing attempt."""
    step_name: str
    error: str
    diagnosis: str
    fix_strategy: str
    confidence: float
    success: bool
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class SelfHealingWrapper:
    """Wraps pipeline steps with automatic error recovery."""

    def __init__(self, max_retries: int = 3, min_confidence: float = 0.6):
        self.max_retries = max_retries
        self.min_confidence = min_confidence
        self.healing_log: list[HealingAttempt] = []

    def _diagnose(self, step_name: str, error: str,
                  context: dict[str, Any]) -> dict:
        """Use LLM to diagnose an error and suggest a fix."""
        raw = mock_llm(
            system_prompt="You are a pipeline error diagnostician. Diagnose and suggest a fix.",
            user_prompt=f"Step '{step_name}' failed with error: {error}\nContext: {json.dumps(context, default=str)[:500]}"
        )
        return json.loads(raw)

    def execute_with_healing(self, step_name: str, fn, *args,
                              context: dict[str, Any] | None = None,
                              **kwargs) -> Any:
        """Execute a function with self-healing retry logic."""
        context = context or {}
        last_error = None

        for attempt in range(self.max_retries + 1):
            try:
                result = fn(*args, **kwargs)
                if attempt > 0:
                    print(f"  [SelfHealing] {step_name}: Recovered on attempt {attempt + 1}")
                return result
            except Exception as e:
                last_error = str(e)
                print(f"  [SelfHealing] {step_name}: Failed (attempt {attempt + 1}): {last_error}")

                if attempt < self.max_retries:
                    # Diagnose and attempt fix
                    diagnosis = self._diagnose(step_name, last_error, context)
                    confidence = diagnosis.get("confidence", 0.0)

                    healing = HealingAttempt(
                        step_name=step_name,
                        error=last_error,
                        diagnosis=diagnosis.get("diagnosis", ""),
                        fix_strategy=diagnosis.get("fix_strategy", ""),
                        confidence=confidence,
                        success=False  # Will update if retry succeeds
                    )
                    self.healing_log.append(healing)

                    if confidence < self.min_confidence:
                        print(f"  [SelfHealing] Confidence too low ({confidence:.2f}), giving up.")
                        break

                    print(f"  [SelfHealing] Fix: {diagnosis.get('fix_strategy', '?')} "
                          f"(confidence={confidence:.2f})")

        raise RuntimeError(
            f"Step '{step_name}' failed after {self.max_retries + 1} attempts. "
            f"Last error: {last_error}"
        )

print("SelfHealingWrapper defined.")

In [ ]:
# --- Test Self-Healing ---

healer = SelfHealingWrapper(max_retries=3)

# A function that fails twice then succeeds
call_count = 0
def flaky_transform(data):
    global call_count
    call_count += 1
    if call_count <= 2:
        raise ValueError(f"Malformed date in row 5 (attempt {call_count})")
    return {"status": "success", "rows": len(data)}

call_count = 0
result = healer.execute_with_healing(
    "date_transform", flaky_transform, sample_data,
    context={"step": "date_parsing"}
)
print(f"\nResult: {result}")
print(f"Healing log: {len(healer.healing_log)} attempts")
for h in healer.healing_log:
    print(f"  {h.step_name}: {h.diagnosis[:60]}... confidence={h.confidence:.2f}")

---
## 6 Pipeline Orchestrator

The **Orchestrator** manages the full ETL pipeline: infer schema, check quality, transform, re-check quality, and output. Each step is wrapped with self-healing.

In [ ]:
class PipelineStage(Enum):
    SCHEMA_INFERENCE = "schema_inference"
    QUALITY_CHECK_PRE = "quality_check_pre"
    TRANSFORM = "transform"
    QUALITY_CHECK_POST = "quality_check_post"
    OUTPUT = "output"
    COMPLETE = "complete"
    FAILED = "failed"


@dataclass
class PipelineRun:
    """State of a pipeline run."""
    run_id: str
    stage: PipelineStage
    input_rows: int = 0
    output_rows: int = 0
    schema: DataSchema | None = None
    quality_pre: QualityReport | None = None
    quality_post: QualityReport | None = None
    transform_result: TransformResult | None = None
    healing_attempts: list[HealingAttempt] = field(default_factory=list)
    errors: list[str] = field(default_factory=list)
    started_at: str = field(default_factory=lambda: datetime.now().isoformat())
    completed_at: str | None = None
    duration_seconds: float = 0.0


class PipelineOrchestrator:
    """Orchestrates the full ETL pipeline."""

    def __init__(self, quality_rules: list[QualityRule],
                 quality_threshold: float = 0.7):
        self.schema_agent = SchemaInferenceAgent()
        self.quality_checker = DataQualityChecker(
            rules=quality_rules, fail_threshold=quality_threshold
        )
        self.transform_agent = TransformAgent()
        self.healer = SelfHealingWrapper(max_retries=3)

    def run(self, data: list[dict[str, str]],
            verbose: bool = True) -> PipelineRun:
        """Execute the full ETL pipeline."""
        run_id = f"RUN-{uuid.uuid4().hex[:8].upper()}"
        state = PipelineRun(run_id=run_id, stage=PipelineStage.SCHEMA_INFERENCE,
                            input_rows=len(data))
        start = time.time()

        # Stage 1: Schema Inference
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Pipeline {run_id} - Stage 1: SCHEMA INFERENCE")
            print(f"{'='*60}")
        try:
            state.schema = self.healer.execute_with_healing(
                "schema_inference", self.schema_agent.infer, data
            )
        except Exception as e:
            state.errors.append(str(e))
            state.stage = PipelineStage.FAILED
            return state

        # Stage 2: Pre-Transform Quality Check
        state.stage = PipelineStage.QUALITY_CHECK_PRE
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Stage 2: PRE-TRANSFORM QUALITY CHECK")
            print(f"{'='*60}")
        state.quality_pre = self.quality_checker.check(data)

        # Stage 3: Transform
        state.stage = PipelineStage.TRANSFORM
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Stage 3: TRANSFORM")
            print(f"{'='*60}")
        try:
            state.transform_result = self.healer.execute_with_healing(
                "transform", self.transform_agent.transform,
                data, state.schema, state.quality_pre
            )
        except Exception as e:
            state.errors.append(str(e))
            state.stage = PipelineStage.FAILED
            return state

        # Stage 4: Post-Transform Quality Check
        state.stage = PipelineStage.QUALITY_CHECK_POST
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Stage 4: POST-TRANSFORM QUALITY CHECK")
            print(f"{'='*60}")
        state.quality_post = self.quality_checker.check(state.transform_result.data)

        # Stage 5: Output
        state.stage = PipelineStage.OUTPUT
        state.output_rows = len(state.transform_result.data)
        if verbose:
            print(f"\n{'='*60}")
            print(f"  Stage 5: OUTPUT")
            print(f"{'='*60}")
            print(f"  Output rows: {state.output_rows}")

        # Complete
        state.stage = PipelineStage.COMPLETE
        state.healing_attempts = list(self.healer.healing_log)
        state.completed_at = datetime.now().isoformat()
        state.duration_seconds = round(time.time() - start, 3)

        if verbose:
            print(f"\n{'='*60}")
            print(f"  PIPELINE COMPLETE ({state.duration_seconds}s)")
            print(f"{'='*60}")
            print(f"  Input rows:     {state.input_rows}")
            print(f"  Output rows:    {state.output_rows}")
            print(f"  Quality before: {state.quality_pre.quality_score:.3f}")
            print(f"  Quality after:  {state.quality_post.quality_score:.3f}")
            print(f"  Healing attempts: {len(state.healing_attempts)}")
            print(f"  Errors: {len(state.errors)}")

        return state

print("PipelineOrchestrator defined.")

---
## 7 Monitoring Dashboard

The **Monitoring Dashboard** tracks pipeline runs and provides metrics and summaries.

In [ ]:
@dataclass
class PipelineMetrics:
    """Aggregated pipeline metrics."""
    total_runs: int
    successful_runs: int
    failed_runs: int
    avg_duration: float
    avg_quality_improvement: float
    total_healing_attempts: int
    total_rows_processed: int


class MonitoringDashboard:
    """Tracks and reports on pipeline runs."""

    def __init__(self):
        self.runs: list[PipelineRun] = []

    def record(self, run: PipelineRun) -> None:
        """Record a pipeline run."""
        self.runs.append(run)

    def get_metrics(self) -> PipelineMetrics:
        """Compute aggregated metrics."""
        successful = [r for r in self.runs if r.stage == PipelineStage.COMPLETE]
        failed = [r for r in self.runs if r.stage == PipelineStage.FAILED]

        durations = [r.duration_seconds for r in successful if r.duration_seconds > 0]
        quality_improvements = []
        for r in successful:
            if r.quality_pre and r.quality_post:
                improvement = r.quality_post.quality_score - r.quality_pre.quality_score
                quality_improvements.append(improvement)

        return PipelineMetrics(
            total_runs=len(self.runs),
            successful_runs=len(successful),
            failed_runs=len(failed),
            avg_duration=round(sum(durations) / max(len(durations), 1), 3),
            avg_quality_improvement=round(
                sum(quality_improvements) / max(len(quality_improvements), 1), 3
            ),
            total_healing_attempts=sum(len(r.healing_attempts) for r in self.runs),
            total_rows_processed=sum(r.input_rows for r in self.runs)
        )

    def display(self) -> None:
        """Display the dashboard."""
        m = self.get_metrics()
        print(f"\n{'='*60}")
        print(f"  PIPELINE MONITORING DASHBOARD")
        print(f"{'='*60}")
        print(f"  Total runs:           {m.total_runs}")
        print(f"  Successful:           {m.successful_runs}")
        print(f"  Failed:               {m.failed_runs}")
        print(f"  Success rate:         {m.successful_runs / max(m.total_runs, 1) * 100:.1f}%")
        print(f"  Avg duration:         {m.avg_duration}s")
        print(f"  Avg quality improve:  {m.avg_quality_improvement:+.3f}")
        print(f"  Healing attempts:     {m.total_healing_attempts}")
        print(f"  Total rows processed: {m.total_rows_processed}")
        print(f"{'='*60}")

        # Run details
        print(f"\n  Recent Runs:")
        for r in self.runs[-5:]:
            status = "OK" if r.stage == PipelineStage.COMPLETE else "FAILED"
            q_pre = r.quality_pre.quality_score if r.quality_pre else "N/A"
            q_post = r.quality_post.quality_score if r.quality_post else "N/A"
            print(f"  {r.run_id} | {status:6s} | {r.duration_seconds:6.3f}s | "
                  f"rows={r.input_rows} | quality: {q_pre} -> {q_post}")

print("MonitoringDashboard defined.")

---
## 8 Full ETL Demo

Run the complete agentic ETL pipeline on the sample data.

In [ ]:
# --- Full ETL Demo ---

orchestrator = PipelineOrchestrator(
    quality_rules=QUALITY_RULES,
    quality_threshold=0.7
)

pipeline_run = orchestrator.run(sample_data)

In [ ]:
# --- Display the monitoring dashboard ---

dashboard = MonitoringDashboard()
dashboard.record(pipeline_run)

# Run a second time with the same data to show multiple runs
orchestrator2 = PipelineOrchestrator(
    quality_rules=QUALITY_RULES, quality_threshold=0.7
)
run2 = orchestrator2.run(sample_data, verbose=False)
dashboard.record(run2)

dashboard.display()

In [ ]:
# --- Inspect transformed data ---

if pipeline_run.transform_result:
    print("Transformed data (first 5 rows):")
    for i, row in enumerate(pipeline_run.transform_result.data[:5]):
        print(f"  Row {i}: {row}")

    print(f"\nQuality improvement: "
          f"{pipeline_run.quality_pre.quality_score:.3f} -> "
          f"{pipeline_run.quality_post.quality_score:.3f}")
else:
    print("No transform result available.")

---
## 9 Domain Variants

The agentic ETL architecture adapts to different domains by customizing schema expectations, quality rules, and transformation logic.

In [ ]:
@dataclass
class ETLDomainConfig:
    """Domain-specific ETL configuration."""
    name: str
    expected_schemas: list[str]
    quality_rules: list[str]
    transform_priorities: list[str]
    compliance_requirements: list[str]

ETL_DOMAINS = {
    "financial": ETLDomainConfig(
        name="Financial Data ETL",
        expected_schemas=["transaction_id", "amount", "currency", "timestamp", "account_id"],
        quality_rules=["Amount must be numeric", "Currency must be ISO 4217",
                       "No duplicate transaction IDs"],
        transform_priorities=["Currency normalization", "Timestamp standardization",
                              "Amount precision (2 decimal places)"],
        compliance_requirements=["SOX compliance", "Audit trail", "PCI-DSS for card data"]
    ),
    "healthcare": ETLDomainConfig(
        name="Healthcare Data ETL",
        expected_schemas=["patient_id", "diagnosis_code", "treatment_date", "provider_id"],
        quality_rules=["Diagnosis codes must be valid ICD-10",
                       "Dates must not be in the future", "No orphan records"],
        transform_priorities=["PHI de-identification", "Code normalization",
                              "Date standardization"],
        compliance_requirements=["HIPAA compliance", "PHI must be encrypted",
                                 "Minimum necessary principle"]
    ),
    "ecommerce": ETLDomainConfig(
        name="E-Commerce Data ETL",
        expected_schemas=["order_id", "product_id", "quantity", "price", "customer_id"],
        quality_rules=["Quantity must be positive", "Price must be positive",
                       "Valid product IDs"],
        transform_priorities=["Price normalization (single currency)",
                              "Product ID deduplication", "Address standardization"],
        compliance_requirements=["GDPR for EU customers", "PCI-DSS for payments"]
    )
}

for name, cfg in ETL_DOMAINS.items():
    print(f"\n{cfg.name}")
    print(f"  Schema:     {cfg.expected_schemas}")
    print(f"  Rules:      {cfg.quality_rules[:2]}...")
    print(f"  Compliance: {cfg.compliance_requirements}")

---
## 10 Exercises

### Exercise 1 (Conceptual)

The current self-healing mechanism uses a simple retry loop with LLM diagnosis. In production, pipelines process millions of rows and failures can be systemic.

**Question:** Design an advanced self-healing system that:
- Categorizes errors into transient (retry), data (fix), and systemic (alert).
- Maintains a knowledge base of past failures and their resolutions.
- Can quarantine bad rows while continuing processing of good data.
- Learns from each failure to prevent the same issue in future runs.

Describe the error taxonomy, quarantine mechanism, and learning loop.

---

### Exercise 2 (Coding)

Implement a `DataProfiler` agent that generates a comprehensive statistical profile of a dataset before and after transformation.

```python
@dataclass
class ColumnProfile:
    name: str
    dtype: str
    null_count: int
    null_pct: float
    unique_count: int
    unique_pct: float
    min_value: str
    max_value: str
    mean: float | None      # For numeric columns
    std_dev: float | None   # For numeric columns
    top_values: list[tuple[str, int]]  # (value, count) pairs

@dataclass
class DataProfile:
    row_count: int
    column_count: int
    columns: list[ColumnProfile]
    completeness: float  # Overall non-null percentage

class DataProfiler:
    def profile(self, data: list[dict[str, str]]) -> DataProfile:
        # Your implementation here
        pass

    def compare(self, before: DataProfile, after: DataProfile) -> str:
        # Generate a comparison report
        pass
```

---

### Exercise 3 (Design)

Design an **agentic data pipeline marketplace** where:
- Users can compose pipelines by chaining pre-built, tested transform agents.
- Each transform agent has a defined input/output schema contract.
- The system validates schema compatibility when chaining agents.
- A supervisor agent recommends pipeline compositions based on data characteristics.

Write a design document covering:
1. Agent interface contract (input schema, output schema, config)
2. Schema compatibility validation
3. Pipeline composition UX
4. Versioning and backward compatibility

In [ ]:
# --- Scratch cell for exercises ---

print("Ready for exercises.")